# Phân tích dữ liệu MIND và tạo vector biểu diễn

Notebook này chuẩn bị dữ liệu tin tức, tạo vector cho từng tin và tổng hợp thành vector của người dùng.

## 1. Thiết lập môi trường

Import các thư viện cần dùng và cấu hình cách hiển thị số.

Import primary library


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display
import json

np.set_printoptions(
    precision=4,  # 4 chữ số thập phân
    suppress=True,  # không dùng dạng 1.23e-05
    linewidth=200,  # tránh xuống dòng quá sớm
)

### Công cụ xử lý dữ liệu

Nạp các thư viện để tải dữ liệu, thao tác tệp và tạo đặc trưng từ văn bản.

In [2]:
import os
import kagglehub
import shutil
from sklearn.feature_extraction.text import CountVectorizer

## 2. Tải và chuẩn bị dữ liệu

Tải bộ MIND, lưu các tệp cần thiết vào thư mục làm việc và tạo mẫu dữ liệu nhỏ.

Load Data


In [3]:
path = kagglehub.dataset_download("arashnic/mind-news-dataset")

print("Dataset downloaded to: ", path)

Dataset downloaded to:  C:\Users\tonmi\.cache\kagglehub\datasets\arashnic\mind-news-dataset\versions\2


Copy data to working dir


In [4]:
source = os.path.join(path, "MINDsmall_train")

destination = r"D:\CDNC\MIND-research\data\raw"

os.makedirs(destination, exist_ok=True)

files = [
    "news.tsv",
    "behaviors.tsv",
    "entity_embedding.vec",
    "relation_embedding.vec",
]

for file in files:
    shutil.copy2(os.path.join(source, file), os.path.join(destination, file))

print("Done")

Done


## 3. Chuẩn bị lịch sử đọc và tin tức

Chọn một nhóm người dùng mẫu, lấy các tin họ đã đọc và lọc thông tin tin tức tương ứng.

**Prepare users behavious dataset**


In [5]:
behaviours_path = os.path.join(destination, "behaviors.tsv")

columns_behaviours = ["user_id", "time", "history", "impressions"]

behaviours = pd.read_csv(
    behaviours_path,
    sep="\t",
    names=columns_behaviours,
)

behaviours = behaviours[["user_id", "history"]]
behaviours = behaviours.dropna(subset=["history"])

sample_behaviours = behaviours.sample(n=10, random_state=42).reset_index(drop=True)

sample_dir = r"D:\CDNC\MIND-research\data\sample"

sample_behaviours.to_csv(os.path.join(sample_dir, "behaviours.csv"), index=False)

print(sample_behaviours.head())

  user_id                                            history
0  U10339          N31739 N12411 N11346 N61388 N12676 N15676
1  U81911  N16082 N38457 N39481 N14734 N21241 N54659 N464...
2  U77463                                             N22345
3  U93135  N26136 N16233 N46978 N32483 N39117 N4020 N3399...
4  U27678  N25691 N63842 N55388 N50155 N47558 N36920 N235...


**Get all readed news in user behaviours dataset**


In [6]:
user_behaviours = sample_behaviours.set_index("user_id")["history"].to_dict()
all_readed_news = set()

for key, value in user_behaviours.items():
    [all_readed_news.add(i) for i in value.split()]

print(len(all_readed_news))

346


**Get news dataset**


In [7]:
news_path = os.path.join(destination, "news.tsv")

if not os.path.exists(news_path):
    f_news_small = open(news_path, "x", encoding="utf-8")


columns = [
    "News_ID",
    "Category",
    "SubCategory",
    "Title",
    "Abstract",
    "URL",
    "Title_Entities",
    "Abstract_Entities",
]

news = pd.read_csv(
    news_path,
    sep="\t",
    names=columns,
)

news = news[["News_ID", "Category", "Title"]]

sample_news = news[news["News_ID"].isin(all_readed_news)].reset_index(drop=True)

sample_dir = r"D:\CDNC\MIND-research\data\sample"

sample_news.to_csv(os.path.join(sample_dir, "news.csv"), index=False)

print(sample_news.shape)
print(sample_news.head())

(346, 3)
  News_ID   Category                                              Title
0  N12676  lifestyle     Pollution around the world: 30 shocking photos
1  N50299         tv  Kelly Ripa responds to backlash over son in 'e...
2  N47341     sports  Al Horford rings his Sixers career in style in...
3  N22505     sports  Cleveland Browns vs. New England Patriots: Wee...
4   N5978    finance  Eric Tse, 24, just became a billionaire overnight


## 4. Khởi tạo mô hình

Khởi tạo mô hình embedding câu, mô hình chủ đề và nơi lưu các kết quả vector.

In [8]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic


class Models:
    def __init__(self):
        self.sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

        self.svectorizer = CountVectorizer(
            stop_words="english",
        )

        self.topic_model = BERTopic(
            calculate_probabilities=True,  # Important to set this to True for probability calculations
            verbose=True,
            vectorizer_model=self.svectorizer,
        )


models = Models()


class VectorContext:
    def __init__(self):
        self.title_list = None
        self.semantic_vector_list = None
        self.probabilities_list = None
        self.topics_vector_list = None

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 5. Tạo vector biểu diễn cho tin tức

Mỗi tin được biểu diễn theo ngữ nghĩa của tiêu đề và phân bố chủ đề.

### Vector Factory

`VectorFactory` được sử dụng để tạo đối tượng biểu diễn vector tương ứng với từng mô hình thông qua một giao diện thống nhất. Thay vì khởi tạo trực tiếp từng lớp, người dùng chỉ cần chỉ định loại mô hình (`sentence` hoặc `bertopic`), Factory sẽ trả về đối tượng phù hợp.

Thiết kế này giúp:

- Tách biệt logic khởi tạo khỏi logic xử lý.
- Dễ dàng thay thế hoặc mở rộng sang các mô hình mới mà không ảnh hưởng đến mã nguồn hiện có.
- Tăng khả năng bảo trì và tái sử dụng mã nguồn.


Vector Interface


In [9]:
from abc import ABC, abstractmethod


class Vector(ABC):
    @abstractmethod
    def get_vector(self):
        pass

    @abstractmethod
    def overview(self):
        pass

    @abstractmethod
    def summary(self):
        pass

Semantic vector


In [10]:
class SentenceVector(Vector):
    def __init__(self, model):
        self.model = model
        self.semantic_vector = None

    def get_vector(self, title_list):
        self.title_list = title_list

        titles = [news["title"] for news in title_list]

        vectors = self.model.encode(
            titles, show_progress_bar=True, convert_to_numpy=True
        )

        self.semantic_vector = {
            news["news_id"]: vector for news, vector in zip(title_list, vectors)
        }
        return self.semantic_vector

    def overview(self):
        print("=" * 60)
        print("Sentence Embedding Overview")
        print(f"[Sentence is: {list(self.semantic_vector.keys())[0]}]")
        print("=" * 60)

        print(f"Documents : {len(self.semantic_vector)}")
        print(
            f"Dimension : {self.semantic_vector[list(self.semantic_vector.keys())[0]].shape[0]}"
        )
        print(
            f"Shape     : {self.semantic_vector[list(self.semantic_vector.keys())[0]].shape}"
        )

        print("\nFirst vector (first 10 values):")
        print(self.semantic_vector[list(self.semantic_vector.keys())[0]][:10], "...")

    def summary(self, sample_index=0):

        sample = self.title_list[sample_index]

        news_id = sample["news_id"]
        title = sample["title"]

        vector = self.semantic_vector[news_id]

        metrics = pd.DataFrame(
            {
                "Metric": [
                    "Embedding Model",
                    "Number of Documents",
                    "Embedding Dimension",
                    "Output Shape",
                ],
                "Value": [
                    self.model.__class__.__name__,
                    len(self.semantic_vector),
                    vector.shape[0],
                    str(vector.shape),
                ],
            }
        )

        vector_preview = ", ".join(f"{x:.4f}" for x in vector[:10]) + ", ..."

        example = pd.DataFrame(
            {
                "News ID": [news_id],
                "Sample Title": [title],
                "Embedding (first 10 dims)": [f"[{vector_preview}]"],
            }
        )

        return metrics, example

Topic vector


In [11]:
class BERTopicVector(Vector):
    def __init__(self, model):
        self.model = model

        self.notice = (
            "BERTopic depends on the sentence transformer model."
            " Please ensure that the sentence transformer model is trained before using BERTopic."
        )

        self.probabilities = None
        self.topics_vector = None

    def get_vector(self, title_list, semantic_vector=None):
        self.title_list = title_list

        titles = [news["title"] for news in title_list]

        embeddings = np.array([semantic_vector[news["news_id"]] for news in title_list])

        if semantic_vector is None:
            print(self.notice)
            return

        topics, probabilities = self.model.fit_transform(
            titles,
            embeddings,
        )

        self.topic_vector = {
            news["news_id"]: {"topic": topic, "probability": probability}
            for news, topic, probability in zip(title_list, topics, probabilities)
        }

        return self.topic_vector

    def overview(self):

        print("=" * 60)
        print("BERTopic Overview")
        print("=" * 60)

        print(f"Number of documents : {len(self.title_list)}")
        print(
            f"Number of topics    : {len(set(v['topic'] for v in self.topic_vector.values()) - {-1})}"
        )

        print("\nTopic distribution:")
        print(self.model.get_topic_info()[["Topic", "Count"]])

        print("\nFirst 5 documents:")

        for sample in self.title_list[:5]:

            news_id = sample["news_id"]
            title = sample["title"]

            topic = self.topic_vector[news_id]["topic"]

            print(f"{news_id}")
            print(f"Title : {title}")
            print(f"Topic : {topic}")
            print("-" * 40)

        first_probability = next(iter(self.topic_vector.values()))["probability"]

        print("\nProbability shape:")
        print(first_probability.shape)

        print("\nFirst 5 probability vectors:")

        for sample in self.title_list[:5]:

            news_id = sample["news_id"]

            probability = self.topic_vector[news_id]["probability"]

            preview = ", ".join(f"{p:.4f}" for p in probability[:10])

            print(f"{news_id} -> [{preview}, ...]")

    def summary(self, sample_index=0):

        # =========================
        # Metrics
        # =========================

        topics = [value["topic"] for value in self.topic_vector.values()]

        first_probability = next(iter(self.topic_vector.values()))["probability"]

        metrics_df = pd.DataFrame(
            {
                "Metric": [
                    "Topic Model",
                    "Number of Documents",
                    "Number of Topics",
                    "Number of Outliers",
                    "Probability Shape",
                ],
                "Value": [
                    self.model.__class__.__name__,
                    len(self.title_list),
                    len(set(topics) - {-1}),
                    np.sum(np.array(topics) == -1),
                    str(first_probability.shape),
                ],
            }
        )

        # =========================
        # Topic Information
        # =========================

        topic_df = self.model.get_topic_info()[["Topic", "Count"]].copy()

        keywords = []

        for topic in topic_df["Topic"]:

            if topic == -1:
                keywords.append("Outlier")
            else:
                words = [word for word, _ in self.model.get_topic(topic)[:5]]
                keywords.append(", ".join(words))

        topic_df["Top Keywords"] = keywords

        # =========================
        # Sample
        # =========================

        sample = self.title_list[sample_index]

        news_id = sample["news_id"]
        title = sample["title"]

        topic_info = self.topic_vector[news_id]

        probs = ", ".join(f"{p:.4f}" for p in topic_info["probability"])

        sample_df = pd.DataFrame(
            {
                "News ID": [news_id],
                "Sample Title": [title],
                "Assigned Topic": [topic_info["topic"]],
                "Probability Distribution": [f"[{probs}]"],
            }
        )

        return metrics_df, topic_df, sample_df

Title list


In [12]:
model_context = VectorContext()
to_dict = lambda news: {"news_id": news["News_ID"], "title": news["Title"]}
model_context.title_list = [to_dict(news) for _, news in sample_news.iterrows()]

print(model_context.title_list)

[{'news_id': 'N12676', 'title': 'Pollution around the world: 30 shocking photos'}, {'news_id': 'N50299', 'title': "Kelly Ripa responds to backlash over son in 'extreme poverty' joke"}, {'news_id': 'N47341', 'title': 'Al Horford rings his Sixers career in style in win over his former Boston Celtics teammates'}, {'news_id': 'N22505', 'title': 'Cleveland Browns vs. New England Patriots: Week 8 TV Listings'}, {'news_id': 'N5978', 'title': 'Eric Tse, 24, just became a billionaire overnight'}, {'news_id': 'N26401', 'title': "Powerful nor'easter brings heavy rain to New York area"}, {'news_id': 'N4643', 'title': "Julianne Hough Mourns the Deaths of Her Two 'Babies' Lexi and Harley: 'I Am Forever Grateful'"}, {'news_id': 'N51238', 'title': 'Why the Patriots made a very un-Patriots trade for Mohamed Sanu'}, {'news_id': 'N22502', 'title': 'Youngest of musical Hanson brothers injured in Tulsa crash'}, {'news_id': 'N27341', 'title': 'These Brides Got Married in the Hawaiian Jungle, and There Was a

Semantic vector


In [13]:
sentence_vector = SentenceVector(models.sentence_model)

model_context.semantic_vector_list = sentence_vector.get_vector(
    model_context.title_list
)
# print(model_context.semantic_vector_list)
sentence_vector.overview()

metric, example = sentence_vector.summary()

print("\nSummary of Sentence Embedding:")
display(metric)
print("\nExample of Sentence Embedding:")
display(example)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Sentence Embedding Overview
[Sentence is: N12676]
Documents : 346
Dimension : 384
Shape     : (384,)

First vector (first 10 values):
[-0.0099  0.084   0.1057  0.0355  0.1892  0.0015  0.0658  0.0067 -0.0461  0.0325] ...

Summary of Sentence Embedding:


,Metric,Value
0,Embedding Model,SentenceTransformer
1,Number of Documents,346
2,Embedding Dimension,384
3,Output Shape,"(384,)"



Example of Sentence Embedding:


,News ID,Sample Title,Embedding (first 10 dims)
0,N12676,Pollution around the world: 30 shocking photos,"[-0.0099, 0.0840, 0.1057, 0.0355, 0.1892, 0.00..."


### Tạo vector chủ đề

Gán chủ đề cho từng tiêu đề và hiển thị bản tóm tắt kết quả.

In [14]:
bertopic_vector = BERTopicVector(models.topic_model)

topics_vector = bertopic_vector.get_vector(
    model_context.title_list, model_context.semantic_vector_list
)
model_context.topics_vector_list = topics_vector
bertopic_vector.overview()

metric, topic, example = bertopic_vector.summary()
print("\nSummary of BERTopic:")
display(metric)
print("\nTopic Information:")
display(topic)
print("\nExample of BERTopic:")
display(example)

2026-07-18 19:17:59,501 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-18 19:18:10,615 - BERTopic - Dimensionality - Completed ✓
2026-07-18 19:18:10,616 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-18 19:18:10,658 - BERTopic - Cluster - Completed ✓
2026-07-18 19:18:10,665 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-18 19:18:10,695 - BERTopic - Representation - Completed ✓


BERTopic Overview
Number of documents : 346
Number of topics    : 2

Topic distribution:
   Topic  Count
0      0    307
1      1     39

First 5 documents:
N12676
Title : Pollution around the world: 30 shocking photos
Topic : 0
----------------------------------------
N50299
Title : Kelly Ripa responds to backlash over son in 'extreme poverty' joke
Topic : 0
----------------------------------------
N47341
Title : Al Horford rings his Sixers career in style in win over his former Boston Celtics teammates
Topic : 0
----------------------------------------
N22505
Title : Cleveland Browns vs. New England Patriots: Week 8 TV Listings
Topic : 1
----------------------------------------
N5978
Title : Eric Tse, 24, just became a billionaire overnight
Topic : 0
----------------------------------------

Probability shape:
(2,)

First 5 probability vectors:
N12676 -> [0.7554, 0.1429, ...]
N50299 -> [0.9080, 0.0920, ...]
N47341 -> [0.4862, 0.1680, ...]
N22505 -> [0.1094, 0.7030, ...]
N5978 -> [0.8

,Metric,Value
0,Topic Model,BERTopic
1,Number of Documents,346
2,Number of Topics,2
3,Number of Outliers,0
4,Probability Shape,"(2,)"



Topic Information:


,Topic,Count,Top Keywords
0,0,307,"says, new, woman, son, killed"
1,1,39,"browns, eagles, trade, kitchens, patriots"



Example of BERTopic:


,News ID,Sample Title,Assigned Topic,Probability Distribution
0,N12676,Pollution around the world: 30 shocking photos,0,"[0.7554, 0.1429]"


Represented vector


In [15]:
class RepresentedVector(Vector):
    def __init__(self, title_list, sentence_dict, bertopic_dict):
        self.title_list = title_list
        self.sentence_dict = sentence_dict
        self.bertopic_dict = bertopic_dict
        self.represented_vector = {}

    def get_vector(self):
        self.represented_vector = {
            news["news_id"]: {
                "title": news["title"],
                "semantic": self.sentence_dict[news["news_id"]],
                # "topic": self.bertopic_dict[news["news_id"]]["topic"],
                "topic_distribution": self.bertopic_dict[news["news_id"]][
                    "probability"
                ],
            }
            for news in self.title_list
        }

        return self.represented_vector

    def overview(self):
        print("=" * 80)
        print("Represented Vector Overview")
        print("=" * 80)

        print(f"Documents           : {len(self.represented_vector)}")

        first_news_id = next(iter(self.represented_vector))

        sample = self.represented_vector[first_news_id]

        print(f"Semantic Dimension  : {len(sample['semantic'])}")
        print(f"Topic Distribution  : {len(sample['topic_distribution'])}")
        print(f"Stored Fields       : {list(sample.keys())}")

        print("=" * 80)

    def preview_vector(vector, preview_dims=4):
        vector = [round(float(x), 4) for x in vector]

        if len(vector) <= preview_dims * 2:
            return vector

        return vector[:preview_dims] + ["..."] + vector[-preview_dims:]

    def summary(self, sample_index=0, preview_dims=4):

        news = self.title_list[sample_index]
        news_id = news["news_id"]

        represented = self.represented_vector[news_id]

        semantic = represented["semantic"]
        probability = represented["topic_distribution"]

        def preview(vector):
            vector = [round(float(x), 4) for x in vector]

            if len(vector) <= preview_dims * 2:
                return vector

            return vector[:preview_dims] + ["..."] + vector[-preview_dims:]

        semantic_preview = preview(semantic)
        probability_preview = preview(probability)

        summary_df = pd.DataFrame(
            {
                "Field": [
                    "News ID",
                    "Title",
                    # "Assigned Topic",
                    "Semantic Dimension",
                    "Topic Distribution Dimension",
                ],
                "Value": [
                    news_id,
                    represented["title"],
                    # represented["topic"],
                    len(semantic),
                    len(probability),
                ],
            }
        )

        represented_preview = {
            news_id: {
                "title": represented["title"],
                "semantic": semantic_preview,
                # "topic": represented["topic"],
                "topic_distribution": probability_preview,
            }
        }

        return summary_df, represented_preview

### Kết hợp các vector

Ghép embedding ngữ nghĩa và phân bố chủ đề thành một biểu diễn cho mỗi tin - Represented Vector.

In [16]:
represented_vector = RepresentedVector(
    model_context.title_list,
    model_context.semantic_vector_list,
    model_context.topics_vector_list,
)

# print(model_context.semantic_vector_list)
# print(model_context.semantic_vector_list)
# print(model_context.topics_vector_list)

represented_vector_list = represented_vector.get_vector()
represented_vector.overview()
represented_vector.summary(sample_index=0, preview_dims=4)

Represented Vector Overview
Documents           : 346
Semantic Dimension  : 384
Topic Distribution  : 2
Stored Fields       : ['title', 'semantic', 'topic_distribution']


(                          Field  \
 0                       News ID   
 1                         Title   
 2            Semantic Dimension   
 3  Topic Distribution Dimension   
 
                                             Value  
 0                                          N12676  
 1  Pollution around the world: 30 shocking photos  
 2                                             384  
 3                                               2  ,
 {'N12676': {'title': 'Pollution around the world: 30 shocking photos',
   'semantic': [-0.0099,
    0.084,
    0.1057,
    0.0355,
    '...',
    0.1182,
    -0.0345,
    -0.0869,
    0.0549],
   'topic_distribution': [0.7554, 0.1429]}})

# User Representation Vector


## Prepare sample data User Representation


In [17]:
user_history = sample_behaviours.set_index("user_id")["history"].to_dict()
print("User mapping: \n", user_history)

User mapping: 
 {'U10339': 'N31739 N12411 N11346 N61388 N12676 N15676', 'U81911': 'N16082 N38457 N39481 N14734 N21241 N54659 N4643 N459 N46039 N26401 N11986 N31215 N39118 N64591 N56353 N24890 N7044 N1066 N43635 N10235 N7242 N53052 N39235 N56415 N11101 N30974 N38298 N2910 N55951 N51396 N51692 N46392 N18275 N4593 N54435 N5978 N32312 N24999 N52403 N63229 N54827 N64756 N11136 N22161 N56399 N7062 N42394 N34287 N24591 N13249 N36353 N40526 N37033 N19347 N54842 N54225 N47173 N57528 N31061 N59306 N13394 N9601 N32852 N23718 N38182 N3757 N6102 N63855 N48699 N27454 N90 N14742 N12608 N32004 N55911 N20619 N12844 N18870 N40442 N36888 N37091 N28879 N54235 N35671 N53017', 'U77463': 'N22345', 'U93135': 'N26136 N16233 N46978 N32483 N39117 N4020 N33998 N33969 N6464 N50299 N22502 N39511 N619 N306 N43142 N28088 N53531 N55556 N4607 N49040', 'U27678': 'N25691 N63842 N55388 N50155 N47558 N36920 N23520 N44251 N13700 N29177 N38118 N1633 N40632 N43749 N38118 N63650 N32683 N11068 N55189 N20530 N10202 N51112 N23669

## Get Represented Vector of 1 user


### Tính vector người dùng

Lấy trung bình vector của các tin trong lịch sử đọc để tạo biểu diễn cho một người dùng.

In [18]:
class URV:
    def __init__(self, represented_vector, user_behaviours):
        self.represented_vector = represented_vector
        self.user_behaviours = user_behaviours
        
    def getURV(self, user_id: str):
        user_history_id = self.user_behaviours[user_id].split()
        
        user_history_vector = [
            self.represented_vector[news_id]
            for news_id in user_history_id
        ]

        user_representation_vector = {
            "semantic": np.mean(
                [v["semantic"] for v in user_history_vector],
                axis=0
            ),
            "topic_distribution": np.mean(
                [v["topic_distribution"] for v in user_history_vector],
                axis=0
            )
        }
        
        assert np.allclose(
            np.mean([v["semantic"] for v in user_history_vector], axis=0),
            sum(v["semantic"] for v in user_history_vector) / len(user_history_vector)
        )
        
        return user_representation_vector
        
        
urv = URV(represented_vector=represented_vector_list, user_behaviours=user_history)
urv.getURV("U10339")

{'semantic': array([ 0.0326,  0.0697,  0.0061,  0.011 ,  0.0433,  0.0178,  0.0161, -0.0146, -0.0055,  0.0382,  0.0708,  0.0258,  0.0077,  0.0021, -0.0219, -0.0112,  0.0252, -0.02  , -0.0497,  0.0109, -0.0221,
         0.017 ,  0.0517, -0.0143,  0.0037, -0.0069, -0.0142,  0.0036,  0.0055, -0.0326,  0.0419, -0.0034, -0.0239,  0.011 ,  0.0258, -0.0179,  0.0575, -0.0112,  0.0178,  0.0312,  0.0553, -0.0228,
         0.0162, -0.0167, -0.0267, -0.01  , -0.0197, -0.0141,  0.0412, -0.008 ,  0.0198,  0.0149, -0.0422,  0.0307, -0.0049, -0.0475, -0.0168,  0.032 , -0.0097,  0.0117, -0.0078,  0.0109, -0.0058,
         0.0241,  0.0234,  0.0413, -0.0104, -0.0418,  0.0322,  0.0246,  0.0635,  0.0227,  0.0167, -0.0097, -0.0517, -0.0215,  0.0406,  0.0005,  0.0457, -0.0315,  0.0374, -0.0687,  0.0122,  0.0045,
         0.0023, -0.0178, -0.054 , -0.0108,  0.0087, -0.0049, -0.0167, -0.0054,  0.0529, -0.0152,  0.0184,  0.0145, -0.0055,  0.0168, -0.0397,  0.0128, -0.0186, -0.0187,  0.0077, -0.0206,  0.0421,
   